# [Colab T4] Ablación proxy: TTT n_aug 16 vs 8 — métricas densas

**Objetivo:** guiar el injerto `ttt8` del pipeline campeón SIN gastar quota de Kaggle.
Modelo proxy: **NVARC Qwen3-2B** (competencia ARC real, 18.33% LB; 2B fp16 cabe en T4).
Por tarea y por brazo (n_aug=16 vs 8): TTT LoRA fresco → medimos **tiempo de TTT**,
**NLL teacher-forced de la respuesta verdadera** (antes/después — métrica DENSA, no exact-match)
y **cell-accuracy** del decode greedy. Diseño PAREADO (mismas tareas en ambos brazos).

⚠️ Los scores absolutos de un 2B no transfieren (medimos 0 exactas en <4B); lo que transfiere
es la señal RELATIVA (¿cuánta calidad de adaptación pierde n_aug=8 y cuánto tiempo ahorra?).

**Uso:** Runtime → T4 GPU → añadir secreto `KAGGLE_API_TOKEN` (Colab: 🔑 Secrets) → Run All
(~75 min). Resultado: `ttt_ablation_result.json` (se imprime y descarga al final).

In [ ]:
%%capture
!pip -q install -U transformers peft accelerate kaggle

In [ ]:
import os, json, time, glob
# token de Kaggle desde Colab Secrets (o pega aqui directamente)
try:
    from google.colab import userdata
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
except Exception:
    assert os.environ.get('KAGGLE_API_TOKEN'), 'define KAGGLE_API_TOKEN'

# datos de la competencia (1 fichero con las 1000 tareas de training + soluciones)
!kaggle competitions download arc-prize-2026-arc-agi-2 -f arc-agi_training_challenges.json -p /content -q
!kaggle competitions download arc-prize-2026-arc-agi-2 -f arc-agi_training_solutions.json -p /content -q
!cd /content && for z in *.zip; do unzip -o -q "$z"; done 2>/dev/null; ls -la /content/*.json | head -3

# modelo NVARC 2B (Kaggle Models)
t0 = time.time()
!kaggle models instances versions download sorokin/qwen3_2b_grids15_sft141/Transformers/bfloat16/1 -p /content/m2b --untar -q
mdir = None
for root, dirs, files in os.walk('/content/m2b'):
    if 'config.json' in files and any(f.endswith('.safetensors') for f in files):
        mdir = root; break
print(f'modelo en {mdir} ({time.time()-t0:.0f}s)')

In [ ]:
# ---- formato de grids del pipeline NVARC/LB33 (identico al campeon) + augmentacion D4xcolor ----
import hashlib, random
import numpy as np

def g2s(grid):
    return '\n'.join(''.join(str(int(c)) for c in row) for row in grid)

def fmt_pair(inp, out):
    return f"<|im_start|>user\n{g2s(inp)}<|im_end|><|im_start|>assistant\n{g2s(out)}<|im_end|>"

def fmt_query(inp):
    return f"<|im_start|>user\n{g2s(inp)}<|im_end|><|im_start|>assistant\n"

def d4(grid, k):
    g = np.array(grid)
    if k >= 4: g = g.T; k -= 4
    return np.rot90(g, k)

def make_ttt_examples(task, n_aug, seed=1):
    """n_aug variantes de los train pairs: transformacion D4 + permutacion de colores + shuffle."""
    rnd = random.Random(seed)
    out = []
    for i in range(n_aug):
        k = rnd.randrange(8)
        perm = list(range(10)); tail = perm[1:]; rnd.shuffle(tail); perm[1:] = tail
        pm = np.array(perm)
        pairs = list(task['train']); rnd.shuffle(pairs)
        text = ''.join(fmt_pair(pm[d4(p['input'], k)], pm[d4(p['output'], k)]) for p in pairs)
        out.append(text)
    return out

# holdout determinista (mismo criterio hash del repo)
def is_holdout(tid):
    return int(hashlib.md5(tid.encode()).hexdigest(), 16) % 1000 < 105

chal = json.load(open('/content/arc-agi_training_challenges.json'))
sols = json.load(open('/content/arc-agi_training_solutions.json'))

def cost(t):
    return sum(len(g['input'])*len(g['input'][0]) + len(g['output'])*len(g['output'][0]) for g in t['train'])

hold = sorted([k for k in chal if is_holdout(k)], key=lambda k: cost(chal[k]))
KEYS = hold[8:20]   # 12 tareas baratas-medias (saltando las 8 mas triviales)
print(f'{len(hold)} holdout; usamos {len(KEYS)}: {KEYS[:4]}...')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, set_peft_model_state_dict, get_peft_model_state_dict

tok = AutoTokenizer.from_pretrained(mdir)
model = AutoModelForCausalLM.from_pretrained(mdir, torch_dtype=torch.float16, device_map='cuda:0')
model.config.use_cache = False
lcfg = LoraConfig(r=64, lora_alpha=32, lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lcfg)
init_state = {k: v.clone() for k, v in get_peft_model_state_dict(model).items()}
print('modelo listo (fp16 + LoRA r=64)')

def nll_of_answer(task_key):
    """NLL teacher-forced de la respuesta VERDADERA del test (metrica densa)."""
    t = chal[task_key]
    prompt = ''.join(fmt_pair(p['input'], p['output']) for p in t['train']) + fmt_query(t['test'][0]['input'])
    ans = g2s(sols[task_key][0]) + '<|im_end|>'
    pi = tok(prompt, return_tensors='pt').input_ids
    ai = tok(ans, return_tensors='pt').input_ids
    ids = torch.cat([pi, ai], 1).to('cuda:0')
    with torch.no_grad():
        logits = model(ids).logits.float()
    lp = torch.log_softmax(logits[0, pi.shape[1]-1:-1], -1)
    return -lp.gather(1, ai[0].unsqueeze(1).to('cuda:0')).mean().item()

def greedy_cell_acc(task_key, max_new=900):
    t = chal[task_key]
    prompt = ''.join(fmt_pair(p['input'], p['output']) for p in t['train']) + fmt_query(t['test'][0]['input'])
    ids = tok(prompt, return_tensors='pt').input_ids.to('cuda:0')
    model.config.use_cache = True
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=False, pad_token_id=tok.eos_token_id)
    model.config.use_cache = False
    txt = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    try:
        pred = [[int(c) for c in line if c.isdigit()] for line in txt.strip().split('\n') if line.strip()]
        gt = sols[task_key][0]
        if not pred or len(pred) != len(gt) or any(len(r) != len(gt[0]) for r in pred):
            return 0.0, False
        p = np.array(pred); g = np.array(gt)
        return float((p == g).mean()), bool((p == g).all())
    except Exception:
        return 0.0, False

In [ ]:
from torch.utils.data import DataLoader

def run_ttt(task_key, n_aug, lr=5e-5):
    """TTT fresco: restaurar LoRA -> entrenar 1 epoca sobre n_aug variantes -> medir."""
    set_peft_model_state_dict(model, {k: v.clone() for k, v in init_state.items()})
    texts = make_ttt_examples(chal[task_key], n_aug)
    enc = [tok(t, return_tensors='pt', truncation=True, max_length=3072).input_ids[0] for t in texts]
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    scaler = torch.cuda.amp.GradScaler()
    model.train()
    t0 = time.time()
    for ids in enc:
        ids = ids.unsqueeze(0).to('cuda:0')
        with torch.cuda.amp.autocast(dtype=torch.float16):
            out = model(ids, labels=ids)
        scaler.scale(out.loss).backward()
        scaler.step(opt); scaler.update(); opt.zero_grad()
    ttt_secs = time.time() - t0
    model.eval()
    return ttt_secs

res = {'arms': {}, 'tasks': KEYS}
for n_aug in (16, 8):
    arm = []
    for i, k in enumerate(KEYS):
        set_peft_model_state_dict(model, {kk: v.clone() for kk, v in init_state.items()})
        model.eval()
        nll0 = nll_of_answer(k)
        ttt_secs = run_ttt(k, n_aug)
        nll1 = nll_of_answer(k)
        acc, exact = greedy_cell_acc(k)
        arm.append({'task': k, 'ttt_secs': round(ttt_secs, 1), 'nll_pre': round(nll0, 4),
                    'nll_post': round(nll1, 4), 'delta_nll': round(nll0 - nll1, 4),
                    'cell_acc': round(acc, 4), 'exact': exact})
        print(f'[n{n_aug} {i+1}/{len(KEYS)}] {k}: ttt={ttt_secs:.0f}s nll {nll0:.3f}->{nll1:.3f} acc={acc:.2f} exact={exact}')
    res['arms'][f'n{n_aug}'] = arm

In [ ]:
import numpy as np
summ = {}
for name, arm in res['arms'].items():
    summ[name] = {
        'ttt_secs_media': round(float(np.mean([a['ttt_secs'] for a in arm])), 1),
        'delta_nll_media': round(float(np.mean([a['delta_nll'] for a in arm])), 4),
        'cell_acc_media': round(float(np.mean([a['cell_acc'] for a in arm])), 4),
        'exactas': sum(a['exact'] for a in arm),
    }
# comparacion pareada por tarea
a16 = {a['task']: a for a in res['arms']['n16']}
a8 = {a['task']: a for a in res['arms']['n8']}
pares = [(a16[k]['delta_nll'] - a8[k]['delta_nll'], a16[k]['ttt_secs'] - a8[k]['ttt_secs']) for k in a16]
summ['pareado'] = {
    'ventaja_adaptacion_n16_vs_n8 (delta_nll, >0 = n16 adapta mejor)': round(float(np.mean([p[0] for p in pares])), 4),
    'ahorro_ttt_de_n8_secs_media': round(float(np.mean([p[1] for p in pares])), 1),
    'tareas_donde_n8_adapta_igual_o_mejor': sum(1 for p in pares if p[0] <= 0),
}
res['resumen'] = summ
json.dump(res, open('/content/ttt_ablation_result.json', 'w'), indent=1)
print(json.dumps(summ, indent=1))
try:
    from google.colab import files
    files.download('/content/ttt_ablation_result.json')
except Exception:
    print('resultado en /content/ttt_ablation_result.json')